In [1]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)

# si_url = https://github.com/gcperformance/service-data/releases/latest/download/si.csv
# si = pd.read_csv(si_url, keep_default_na=False, na_values='', delimiter=';')

si_path = parent_dir / 'outputs' / 'si.csv'
si = pd.read_csv(si_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

ss_path = parent_dir / 'outputs' / 'ss.csv'
ss = pd.read_csv(ss_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

In [ ]:
# RCP 1: % of services having used client feedback to improve / review services in the year / 5 yrs prior to reporting
rcp1 = si.loc[:,['fiscal_yr','org_id','service_id', 'department_en', 'department_fr', 'last_service_review', 'last_service_improvement']]

rcp1['report_yr'] = pd.to_numeric(rcp1['fiscal_yr'].str.split('-').str[1], errors='coerce').astype(int)

rcp1['last_service_review_yr'] = pd.to_numeric(rcp1['last_service_review'].str.split('-').str[1], errors='coerce')
rcp1['yrs_since_last_service_review'] = rcp1['report_yr']-rcp1['last_service_review_yr']
rcp1['last_service_review_within_5_yrs'] = (rcp1['yrs_since_last_service_review'] <= 5) & (rcp1['yrs_since_last_service_review'] >= 0)

rcp1['last_service_improvement_yr'] = pd.to_numeric(rcp1['last_service_improvement'].str.split('-').str[1], errors='coerce')
rcp1['yrs_since_last_service_improvement'] = rcp1['report_yr']-rcp1['last_service_improvement_yr']
rcp1['last_service_improvement_within_1_yr'] = (rcp1['yrs_since_last_service_improvement'] <= 1) & (rcp1['yrs_since_last_service_improvement'] >= 0)

rcp1 = rcp1.groupby(['fiscal_yr', 'org_id','department_en', 'department_fr']).agg(
    service_count_rcp1 = ('service_id', 'count'),
    services_reviewed_in_past_5_yrs = ('last_service_review_within_5_yrs', 'sum'),
    services_improved_in_past_1_yr = ('last_service_improvement_within_1_yr', 'sum')
    ).reset_index()

rcp1['rcp1_service_review_pc'] = (rcp1['services_reviewed_in_past_5_yrs']/rcp1['service_count_rcp1'])*100
rcp1['rcp1_service_improvement_pc'] = (rcp1['services_improved_in_past_1_yr']/rcp1['service_count_rcp1'])*100

# RCP 2: % of services that have a service standard for all active channels

# Dataframe aligning service standard channels to service inventory volume channels
channels_df = [
    ['num_applications_by_phone', 'TEL', 'Telephone'],
    ['num_applications_online', 'ONL', 'Online'],
    ['num_applications_in_person', 'PERSON', 'In person'],
    ['num_applications_by_mail', 'POST', 'Postal mail'],
    ['num_applications_by_email', 'EML', 'Email'],
    ['num_applications_by_fax', 'FAX', 'Fax'],
    ['num_applications_by_other', 'OTH', 'Other']
]

channels_df = pd.DataFrame(channels_df, columns=['si_volume_channel', 'ss_channel', 'channel_en'])

# si_channels: channels with volume >0 by service, i.e. active channels
si_channels = (
    si
    .melt(
        id_vars=['fiscal_yr','org_id', 'service_id'], 
        value_vars=channels_df['si_volume_channel'],
        var_name='si_volume_channel',
        value_name='volume')
    .assign(volume=lambda x: pd.to_numeric(x['volume'], errors='coerce')) #convert all volume records to a number
    .merge(channels_df[['si_volume_channel', 'ss_channel']], on='si_volume_channel', how='left') #merges in ss_channels
    .loc[lambda x: x['volume']>0]
)

# ss_channels: which service-channels have defined service standards
ss_channels = (
    ss
    .loc[:, ['fiscal_yr', 'org_id', 'service_id', 'channel']]
    .drop_duplicates()
)

# Join active service-channels from si to service standard channels
rcp2_long = (
    si_channels
    .merge(
        ss_channels,
        how='left',
        left_on=['fiscal_yr', 'org_id', 'service_id', 'ss_channel'],
        right_on=['fiscal_yr', 'org_id', 'service_id', 'channel'])
    .drop(columns=["ss_channel", "volume"])
)

# Group and summarize the distinct service-channel records at the service level, counting the distinct channels from SI and SS
rcp2_service = (
    rcp2_long
    .groupby(['fiscal_yr', 'org_id', 'service_id'], as_index=False)
    .agg(
        active_channels = ('si_volume_channel', 'nunique'),
        channels_with_ss = ('channel', 'nunique'))
    )

# If there is the same number of channels, then all the active channels have a standard, given the earlier merge.
# The merge ensures that there are no false positives through services having the same number of channels in si and ss without them being the same channel 
rcp2_service['all_channels_have_standard'] = (rcp2_service['active_channels'] == rcp2_service['channels_with_ss'])

# Group and summarize by organization the number of services that have full coverage
rcp2 = (
    rcp2_service
    .groupby(['fiscal_yr', 'org_id'], as_index=False)
    .agg(
        services_with_standards_for_all_channels = ('all_channels_have_standard', 'sum')
    )
)

# Determine total services - note that we can't take the aggregate from rcp2 since services without volumes have been excluded from the counts
# We need services without volumes since even if there are no
service_count_by_fy_org = si.groupby(['fiscal_yr', 'org_id','department_en', 'department_fr'], as_index=False).agg(
    service_count_rcp2=('service_id', 'nunique')  # Count all services
)

# merge in service count to rcp2
rcp2 = (
    rcp2
    .merge(
        service_count_by_fy_org,
        how='outer',
        on=['fiscal_yr', 'org_id']
    )
    .fillna(0)
)

rcp2['rcp2_services_with_standards_for_all_channels_pc'] = (rcp2['services_with_standards_for_all_channels']/rcp2['service_count_rcp2'])*100

rcp_all = pd.merge(
    rcp1,
    rcp2,
    how='outer',
    on=['fiscal_yr', 'org_id','department_en', 'department_fr']
).fillna(0)
